# 창문/외벽 인스턴스 분할 학습 (YOLO11s-seg, Colab)

0=window, 1=facade. 데이터는 로컬에서 `merge_datasets`로 만든 디렉토리를 zip해 드라이브에 올린 것.
사용법은 `cv/notebooks/README.md`.

1. Stage1 — CMP만으로 학습 (imgsz 1024, 조기종료)
2. Stage2 — Stage1 가중치로 CMP + 자체 데이터 파인튜닝 (자체 train ×3 오버샘플링)
3. 자체 test셋 평가 — window mask mAP50 / mAP50-95, 창문 개수 정확도, 예측 오버레이 16장
4. best.pt → 드라이브 `weights/seg_window_facade_v{N}.pt`, `weights/MODEL_CARD.md`에 기록 (로컬은 `cv/weights/`에 둔다)

`STAGE1_ONLY = True`면 자체 데이터 없이 1 → CMP val 평가 → 4만 한다 (Stage2, 자체 test 평가는 건너뜀).

## 0. 하이퍼파라미터 — 이 셀만 고치면 된다

In [ ]:
# ── 경로 (Google Drive) ──
DRIVE_ROOT = "/content/drive/MyDrive/facade-inspect"
DATASET_ZIP = f"{DRIVE_ROOT}/datasets/merged.zip"  # merge_datasets 출력 디렉토리를 zip
WEIGHTS_DIR = f"{DRIVE_ROOT}/weights"              # 최종 가중치 + MODEL_CARD.md (로컬 cv/weights/로 받는다)
RUNS_DIR = f"{DRIVE_ROOT}/runs"                    # 학습 결과(project) — 드라이브라 세션이 끊겨도 남는다
WORK_DIR = "/content/data"                         # 압축 해제 위치 (로컬 디스크가 드라이브보다 빠름)

# ── 이어서 학습 ──
RESUME = False    # True: RUN_NAME 실행을 이어간다 — 끝난 Stage는 건너뛰고, 중단된 Stage는 weights/last.pt부터 재개
RUN_NAME = None   # None: 새 실행이면 현재 시각, RESUME이면 RUNS_DIR에서 가장 최근 실행. 특정 실행은 이름 지정

# ── 학습 범위 ──
STAGE1_ONLY = False  # True: 자체 데이터 없이 CMP만 — Stage1 학습 → CMP val 평가 → 저장. Stage2·자체 test 평가는 건너뜀
                     #       (DATASET_ZIP은 자체 데이터 없이 만든 zip이어도 된다. RESUME과 같이 써도 됨)

ULTRALYTICS_VERSION = "8.4.171"   # 로컬 cv/uv.lock과 맞춤
ALBUMENTATIONS_VERSION = "2.0.8"

CLASSES = ["window", "facade"]    # config/dataset.yaml names와 같아야 함
WINDOW_ID = 0
SEED = 0

# ── Stage1: CMP만 ──
STAGE1_WEIGHTS = None             # 이미 학습한 Stage1 best.pt 경로를 넣으면 Stage1을 건너뛴다
STAGE1 = dict(model="yolo11s-seg.pt", imgsz=1024, epochs=100, patience=20, batch=0.7, lr0=0.01, cos_lr=True)

# ── Stage2: CMP + 자체 (자체 train ×OWN_OVERSAMPLE) ──
OWN_OVERSAMPLE = 3
STAGE2 = dict(imgsz=1024, epochs=50, patience=15, batch=0.7, lr0=0.002, warmup_epochs=1, cos_lr=True)
# batch: 0.7 = GPU 메모리의 70%를 쓰는 배치 크기를 자동 계산 (Ultralytics AutoBatch, 단일 CUDA GPU 전용 —
#        CPU/MPS에서는 16으로 대체). 고정하려면 정수로: 예) T4(16GB)·imgsz 1024면 batch=8부터, OOM이면 줄인다.
#        재개(RESUME) 시에는 처음 계산된 배치 크기가 체크포인트에서 그대로 쓰인다

# ── 증강: config/augment.yaml과 동일하게 유지 (cv/tests/test_notebook.py가 확인) ──
AUGMENT = {
    "hsv_v": 0.4, "hsv_s": 0.5, "hsv_h": 0.015,
    "perspective": 0.0005, "degrees": 5.0, "shear": 2.0, "translate": 0.1, "scale": 0.5,
    "fliplr": 0.5, "flipud": 0.0, "mosaic": 1.0, "close_mosaic": 10,
    "albumentations": [
        {"name": "RandomBrightnessContrast", "brightness_limit": 0.0, "contrast_limit": 0.3, "p": 0.5},
        {"name": "MotionBlur", "blur_limit": [3, 11], "p": 0.3},
    ],
}

# ── 평가 ──
MAX_DET = 500       # CMP에 창문 300개 넘는 이미지가 있어 기본 300보다 크게
EVAL_BATCH = 16     # 평가는 AutoBatch(소수 batch)를 지원하지 않아 정수
COUNT_CONF = 0.4    # 창문 개수 정확도 / 오버레이에 쓰는 검출 confidence
COUNT_TOL = 0.1     # |검출-GT|/GT ≤ 10%면 "정확"
GRID_N = 16

## 1. 환경 — 드라이브 마운트, 패키지 설치

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
!pip install -q "ultralytics=={ULTRALYTICS_VERSION}" "albumentations=={ALBUMENTATIONS_VERSION}"

In [ ]:
import torch

# facade_cv.device.get_device와 같은 우선순위 (cuda > mps > cpu). Ultralytics는 cuda를 0으로 받는다
DEVICE = 0 if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else str(DEVICE)
print("device:", DEVICE, GPU_NAME)
if DEVICE != 0:
    print("⚠️ GPU 없음 — 런타임 > 런타임 유형 변경 > GPU")

## 2. 헬퍼 함수 (Colab 의존 없음 — `cv/tests/test_notebook.py`가 로컬에서 실행해 검증)

In [ ]:
import json
import re
import shutil
import time
import zipfile
from pathlib import Path

import numpy as np
import yaml

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}


def is_cmp(img: Path) -> bool:
    return Path(img).name.startswith("cmp__")  # merge_datasets 출력 파일명 규칙


def label_of(img: Path) -> Path:
    """.../images/<split>/x.jpg → .../labels/<split>/x.txt (Ultralytics와 같은 규칙: 마지막 images 디렉토리)."""
    parts = list(Path(img).parts)
    i = len(parts) - 1 - parts[::-1].index("images")
    parts[i] = "labels"
    return Path(*parts).with_suffix(".txt")


def make_stage_yamls(root: Path, oversample: int, names: list[str], require_own: bool = True) -> dict:
    """merge_datasets 출력(root)에서 Stage별 이미지 목록(txt)과 data yaml 생성.

    stage1: train/val = CMP / stage2: train = CMP + 자체×oversample, val = CMP + 자체 / test = 자체 test (둘 다)
    목록에 같은 경로를 여러 번 쓰면 Ultralytics가 그만큼 반복 샘플링한다 (중복 제거 안 함).
    자체 test가 없으면 require_own=True일 때 에러, False(STAGE1_ONLY)면 stage1 yaml만 test 없이 만든다.
    """
    root = Path(root)
    imgs = {s: sorted(p for p in (root / "images" / s).glob("*") if p.suffix.lower() in IMAGE_EXTS)
            for s in ("train", "val", "test")}
    cmp = {s: [p for p in v if is_cmp(p)] for s, v in imgs.items()}
    own = {s: [p for p in v if not is_cmp(p)] for s, v in imgs.items()}
    if not own["test"] and require_own:
        raise ValueError("자체 데이터 test셋이 비어 있음 — merge_datasets --own으로 만든 zip인지 확인 "
                         "(자체 데이터 없이 학습하려면 STAGE1_ONLY = True)")
    if not cmp["train"] or not cmp["val"]:
        raise ValueError("CMP train/val이 비어 있음 — merge_datasets --cmp로 만든 zip인지 확인")
    lists = {"stage1_train": cmp["train"], "stage1_val": cmp["val"]}
    if own["test"]:
        lists.update(stage2_train=cmp["train"] + own["train"] * oversample, stage2_val=cmp["val"] + own["val"],
                     own_test=own["test"])
    (root / "lists").mkdir(exist_ok=True)
    for k, v in lists.items():
        (root / "lists" / f"{k}.txt").write_text("".join(f"{p.resolve()}\n" for p in v))
    out = {"counts": {k: len(v) for k, v in lists.items()}}
    for stage in ("stage1", "stage2") if own["test"] else ("stage1",):
        data = {"path": str(root.resolve()), "train": f"lists/{stage}_train.txt", "val": f"lists/{stage}_val.txt",
                **({"test": "lists/own_test.txt"} if own["test"] else {}), "names": dict(enumerate(names))}
        out[stage] = root / f"data_{stage}.yaml"
        out[stage].write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True))
    return out


def resolve_run_name(runs_dir: Path, run_name: str | None, resume: bool) -> str:
    """지정 이름 > (RESUME이면) 가장 최근 <name>_stage1/2 실행 > 현재 시각."""
    if run_name:
        return run_name
    if not resume:
        return time.strftime("%Y%m%d_%H%M")
    runs = [p for p in Path(runs_dir).glob("*_stage[12]") if p.is_dir()]
    if not runs:
        raise FileNotFoundError(f"RESUME=True인데 {runs_dir}에 이어갈 실행(<RUN_NAME>_stage1/2)이 없음")
    newest = max(runs, key=lambda p: max((f.stat().st_mtime for f in p.rglob("*")), default=p.stat().st_mtime))
    return newest.name.rsplit("_stage", 1)[0]


def stage_state(run_dir: Path) -> str:
    """"new" (last.pt 없음) / "partial" (중단) / "done" (끝남 — Ultralytics가 마지막에 epoch=-1로 저장)."""
    last = Path(run_dir) / "weights" / "last.pt"
    if not last.exists():
        return "new"
    import torch

    ckpt = torch.load(last, map_location="cpu", weights_only=False)
    return "done" if ckpt.get("epoch", -1) == -1 else "partial"


def train_stage(stage: str, init_weights, data: Path, kwargs: dict, runs_dir: str, run_name: str, resume: bool) -> Path:
    """<runs_dir>/<run_name>_<stage>에서 학습. resume이면 끝난 Stage는 건너뛰고 중단된 Stage는 last.pt부터 재개.
    best.pt 경로를 반환."""
    from ultralytics import YOLO

    run_dir = Path(runs_dir) / f"{run_name}_{stage}"
    state = stage_state(run_dir)
    if state != "new" and not resume:
        raise FileExistsError(f"{run_dir}에 학습 결과가 있음 — RESUME=True로 이어가거나 RUN_NAME을 바꿀 것")
    if state == "done":
        best = run_dir / "weights" / "best.pt"
        print(f"{stage}: 이미 끝남 → 건너뜀 ({best})")
        return best if best.exists() else run_dir / "weights" / "last.pt"
    if state == "partial":
        print(f"{stage}: {run_dir / 'weights' / 'last.pt'}부터 재개")
        model = YOLO(str(run_dir / "weights" / "last.pt"))
        model.train(resume=True)  # 설정·증강·배치는 체크포인트에 저장된 값 그대로
    else:
        model = YOLO(str(init_weights))
        model.train(data=str(data), project=str(runs_dir), name=run_dir.name, exist_ok=True, **kwargs)
    return Path(model.trainer.best)


def build_augment(aug: dict) -> dict:
    """AUGMENT → Ultralytics train 인자 (albumentations 항목은 `augmentations` 객체 리스트로)."""
    import albumentations as A

    args = {k: v for k, v in aug.items() if k != "albumentations"}
    args["augmentations"] = [getattr(A, s["name"])(**{k: v for k, v in s.items() if k != "name"})
                             for s in aug.get("albumentations", [])]
    return args


def window_mask_metrics(metrics, window_id: int) -> dict:
    """Ultralytics SegmentMetrics → window 클래스 mask P/R/mAP50/mAP50-95."""
    i = list(metrics.ap_class_index).index(window_id)
    p, r, ap50, ap = metrics.seg.class_result(i)
    return {"mask_P": float(p), "mask_R": float(r), "mask_mAP50": float(ap50), "mask_mAP50-95": float(ap)}


def gt_count(label: Path, cls: int) -> int:
    label = Path(label)
    if not label.exists():
        return 0
    return sum(1 for line in label.read_text().splitlines() if line.split() and int(line.split()[0]) == cls)


def count_metrics(gt: list[int], pred: list[int], tol: float) -> dict:
    """이미지당 창문 개수 오차. rel = |pred-gt| / max(gt, 1)."""
    gt, pred = np.asarray(gt, float), np.asarray(pred, float)
    err = pred - gt
    rel = np.abs(err) / np.maximum(gt, 1)
    return {"images": len(gt), "MAE": float(np.abs(err).mean()), "bias": float(err.mean()),
            "MAPE": float(rel.mean()), f"within_{tol:.0%}": float((rel <= tol).mean()),
            "exact": float((err == 0).mean()), "gt_total": int(gt.sum()), "pred_total": int(pred.sum())}


def next_version(models_dir: Path, prefix: str = "seg_window_facade_v") -> int:
    nums = [int(m.group(1)) for p in Path(models_dir).glob(f"{prefix}*.pt")
            if (m := re.fullmatch(re.escape(prefix) + r"(\d+)\.pt", p.name))]
    return max(nums, default=0) + 1


def saved_marker(run_name: str, final_stage: str) -> str:
    """MODEL_CARD.md에서 같은 실행·같은 최종 Stage의 중복 저장을 막는 표식.
    STAGE1_ONLY로 저장한 실행을 나중에 Stage2까지 이어서 학습하면 최종 Stage가 달라 다시 저장된다."""
    return f"실행: run `{run_name}`, 최종 `{final_stage}`"


def model_card_entry(info: dict) -> str:
    """MODEL_CARD.md에 append할 마크다운 섹션."""
    tol_key = f"within_{info['count_tol']:.0%}"
    rows = "\n".join(
        f"| {k} | {v['mask_mAP50']:.4f} | {v['mask_mAP50-95']:.4f} | {c['MAE']:.2f} | {c['MAPE']:.1%} | {c[tol_key]:.1%} |"
        for k, v in info["eval"].items() for c in [info["count"][k]]
    )
    stage2 = (f"{json.dumps(info['stage2'], ensure_ascii=False)}, 자체 train ×{info['oversample']}"
              if info["final_stage"] == "stage2" else "없음 (STAGE1_ONLY — CMP만)")
    return f"""
## {info['name']} — {info['date']}

- 파일: `{info['file']}` ({info['final_stage']} best.pt)
- 데이터: `{info['dataset']}` — {json.dumps(info['counts'], ensure_ascii=False)}
- Stage1: {json.dumps(info['stage1'], ensure_ascii=False)}
- Stage2: {stage2}
- 증강: `{json.dumps(info['augment'], ensure_ascii=False)}`
- 환경: ultralytics {info['ultralytics']}, {info['gpu']}, seed {info['seed']}
- {saved_marker(info['run'], info['final_stage'])}

평가: {info['eval_set']} {info['eval_images']}장, window 클래스. 개수 정확도는 conf ≥ {info['count_conf']}:

| 모델 | mask mAP50 | mask mAP50-95 | 개수 MAE | 개수 MAPE | ±{info['count_tol']:.0%} 이내 |
|---|---|---|---|---|---|
{rows}
"""

## 3. 데이터 — zip 압축 해제, Stage별 목록 생성

In [ ]:
RUN_NAME = resolve_run_name(RUNS_DIR, RUN_NAME, RESUME)
print("run:", RUN_NAME, "(이어서)" if RESUME else "(새로)")

work = Path(WORK_DIR) / Path(DATASET_ZIP).stem  # zip마다 따로 — 같은 세션에서 zip을 바꿔도 섞이지 않게
if not list(work.rglob("data.yaml")):
    work.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP) as z:
        z.extractall(work)
DATA_ROOT = next(work.rglob("data.yaml")).parent  # zip 안에 상위 폴더가 있든 없든
YAMLS = make_stage_yamls(DATA_ROOT, OWN_OVERSAMPLE, CLASSES, require_own=not STAGE1_ONLY)

# 평가 대상: 기본은 자체 test (Stage1 vs Stage2), STAGE1_ONLY면 CMP val (Stage1만)
FINAL_STAGE = "stage1" if STAGE1_ONLY else "stage2"
EVAL_YAML, EVAL_SPLIT, EVAL_LIST, EVAL_SET = (
    (YAMLS["stage1"], "val", "stage1_val", "CMP val") if STAGE1_ONLY else (YAMLS["stage2"], "test", "own_test", "자체 test")
)
EVAL_IMAGES = [Path(p) for p in (DATA_ROOT / "lists" / f"{EVAL_LIST}.txt").read_text().split()]
print(DATA_ROOT, "| 최종:", FINAL_STAGE, "| 평가:", EVAL_SET, len(EVAL_IMAGES), "장")
YAMLS["counts"]

## 4. Stage1 — CMP만으로 학습

In [ ]:
from ultralytics import YOLO

TRAIN_COMMON = dict(seed=SEED, device=DEVICE, max_det=MAX_DET, **build_augment(AUGMENT))
if STAGE1_WEIGHTS:
    STAGE1_BEST = Path(STAGE1_WEIGHTS)
    print("Stage1 건너뜀:", STAGE1_BEST)
else:
    s1 = dict(STAGE1)
    init = s1.pop("model")
    STAGE1_BEST = train_stage("stage1", init, YAMLS["stage1"], {**TRAIN_COMMON, **s1}, RUNS_DIR, RUN_NAME, RESUME)
print(STAGE1_BEST)

## 5. Stage2 — CMP + 자체 데이터 파인튜닝 (STAGE1_ONLY면 건너뜀)

In [ ]:
if STAGE1_ONLY:
    STAGE2_BEST = None
    print("STAGE1_ONLY — Stage2 건너뜀")
else:
    STAGE2_BEST = train_stage("stage2", STAGE1_BEST, YAMLS["stage2"], {**TRAIN_COMMON, **STAGE2}, RUNS_DIR, RUN_NAME, RESUME)
FINAL_BEST = STAGE2_BEST if FINAL_STAGE == "stage2" else STAGE1_BEST
print(FINAL_STAGE, FINAL_BEST)

## 6. 평가 — window mask mAP (기본: 자체 test, Stage1 vs Stage2 / STAGE1_ONLY: CMP val, Stage1)

In [ ]:
import pandas as pd

WEIGHTS = {"stage1": STAGE1_BEST} if STAGE1_ONLY else {"stage1": STAGE1_BEST, "stage2": STAGE2_BEST}
EVAL = {}
for name, w in WEIGHTS.items():
    m = YOLO(str(w)).val(data=str(EVAL_YAML), split=EVAL_SPLIT, imgsz=STAGE2["imgsz"], batch=EVAL_BATCH,
                         device=DEVICE, max_det=MAX_DET, plots=False, project=RUNS_DIR,
                         name=f"{RUN_NAME}_eval/{name}", exist_ok=True)
    EVAL[name] = window_mask_metrics(m, WINDOW_ID)
print("평가:", EVAL_SET)
pd.DataFrame(EVAL).T

## 7. 창문 개수 정확도 — 이미지당 GT 개수 대비 검출 개수 (6과 같은 평가 대상)

In [ ]:
COUNT, PER_IMAGE = {}, {}
for name, w in WEIGHTS.items():
    rows = []
    for res in YOLO(str(w)).predict([str(p) for p in EVAL_IMAGES], conf=COUNT_CONF, imgsz=STAGE2["imgsz"],
                                     device=DEVICE, max_det=MAX_DET, stream=True, verbose=False):
        rows.append({"image": Path(res.path).name, "gt": gt_count(label_of(res.path), WINDOW_ID),
                     "pred": int((res.boxes.cls == WINDOW_ID).sum())})
    PER_IMAGE[name] = pd.DataFrame(rows)
    COUNT[name] = count_metrics(PER_IMAGE[name]["gt"], PER_IMAGE[name]["pred"], COUNT_TOL)
print(pd.DataFrame(COUNT).T.to_string())
df = PER_IMAGE[FINAL_STAGE].assign(err=lambda d: d["pred"] - d["gt"])  # d.gt는 DataFrame.gt 메서드라 [] 사용
df.reindex(df["err"].abs().sort_values(ascending=False).index).head(10)  # 오차 큰 이미지

## 8. 예측 오버레이 16장 (최종 모델, 평가 대상 → 부족하면 자체 val로 채움)

In [ ]:
import cv2
import matplotlib.pyplot as plt

own_val = [] if STAGE1_ONLY else [Path(p) for p in (DATA_ROOT / "lists" / "stage2_val.txt").read_text().split()
                                  if not is_cmp(p)]
vis = (EVAL_IMAGES + own_val)[:GRID_N]
cols = 4
rows_n = max(1, -(-len(vis) // cols))
fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 5, rows_n * 5), squeeze=False)
results = YOLO(str(FINAL_BEST)).predict([str(p) for p in vis], conf=COUNT_CONF, imgsz=STAGE2["imgsz"],
                                         device=DEVICE, max_det=MAX_DET, verbose=False)
for ax, res in zip(axes.flat, results):
    n_pred = int((res.boxes.cls == WINDOW_ID).sum())
    ax.imshow(cv2.cvtColor(res.plot(boxes=False), cv2.COLOR_BGR2RGB))
    ax.set_title(f"{Path(res.path).name[:28]}\nGT {gt_count(label_of(res.path), WINDOW_ID)} / pred {n_pred}", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
GRID_PATH = Path(RUNS_DIR) / f"{RUN_NAME}_pred_grid.jpg"
fig.savefig(GRID_PATH, dpi=80)
plt.show()

## 9. 저장 — 최종 best.pt → `weights/seg_window_facade_v{N}.pt` + MODEL_CARD.md append

In [ ]:
import ultralytics

weights_dir = Path(WEIGHTS_DIR)
weights_dir.mkdir(parents=True, exist_ok=True)
card = weights_dir / "MODEL_CARD.md"
if card.exists() and saved_marker(RUN_NAME, FINAL_STAGE) in card.read_text(encoding="utf-8"):
    raise SystemExit(f"run {RUN_NAME} ({FINAL_STAGE})은 이미 저장됨 (MODEL_CARD.md) — 같은 모델 중복 저장 방지로 멈춤")
N = next_version(weights_dir)
dst = weights_dir / f"seg_window_facade_v{N}.pt"
shutil.copy2(FINAL_BEST, dst)

entry = model_card_entry({
    "name": dst.stem, "date": time.strftime("%Y-%m-%d %H:%M"), "file": dst.name, "dataset": Path(DATASET_ZIP).name,
    "counts": YAMLS["counts"], "stage1": STAGE1 if not STAGE1_WEIGHTS else {"weights": str(STAGE1_WEIGHTS)},
    "stage2": STAGE2, "oversample": OWN_OVERSAMPLE, "augment": AUGMENT, "ultralytics": ultralytics.__version__,
    "gpu": GPU_NAME, "seed": SEED, "run": RUN_NAME, "eval": EVAL, "count": COUNT,
    "count_conf": COUNT_CONF, "count_tol": COUNT_TOL, "final_stage": FINAL_STAGE,
    "eval_set": EVAL_SET, "eval_images": len(EVAL_IMAGES),
})
if not card.exists():
    card.write_text("# 창문/외벽 분할 모델 카드\n\n0=window, 1=facade. 평가 대상은 항목마다 표기 (자체 test 또는 CMP val).\n"
                    "가중치 위치: 드라이브 `facade-inspect/weights/`, 로컬 `cv/weights/` (config/detect.yaml `weights`).\n")
with card.open("a", encoding="utf-8") as f:
    f.write(entry)
print("saved", dst)
print(entry)